In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Create Spark Session
spark = SparkSession.builder \
    .appName("OrderBy_DropDuplicates") \
    .master("local[*]") \
    .getOrCreate()

# Sample data
data = [
    ("101", "2023-12-01", 100),
    ("101", "2023-12-02", 150),
    ("102", "2023-12-01", 200),
    ("102", "2023-12-02", 250)
]

columns = ["product_id", "date", "sales"]

df = spark.createDataFrame(data, columns)
print("Original Data:")
df.show()
# Sort by product_id ascending and date descending
result = (
    df.orderBy("product_id", F.col("date").desc())
      .dropDuplicates(["product_id"])
)
print("Latest record per product:")
result.show()

spark.stop()

Original Data:
+----------+----------+-----+
|product_id|      date|sales|
+----------+----------+-----+
|       101|2023-12-01|  100|
|       101|2023-12-02|  150|
|       102|2023-12-01|  200|
|       102|2023-12-02|  250|
+----------+----------+-----+

Latest record per product:
+----------+----------+-----+
|product_id|      date|sales|
+----------+----------+-----+
|       101|2023-12-02|  150|
|       102|2023-12-02|  250|
+----------+----------+-----+



In [1]:
print("hhxgh")

hhxgh


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum as _sum

spark = SparkSession.builder \
    .appName("DAG Example") \
    .getOrCreate()

# Create sample data
data = [
    ("A", 10),
    ("A", 20),
    ("B", 15),
    ("B", 25),
    ("C", 30),
    ("C", 35),
    ("D", 40),
    ("D", 45)
]

df = spark.createDataFrame(data, ["category", "value"])

# Force more partitions so shuffle is visible
spark.conf.set("spark.sql.shuffle.partitions", 8)

# Step 1: Filter (narrow transformation – no shuffle)
filtered_df = df.filter(col("value") > 15)

# Step 2: Aggregation (WIDE transformation – causes Exchange)
agg_df = filtered_df.groupBy("category") \
                     .agg(
                         count("*").alias("cnt"),
                         _sum("value").alias("total_value")
                     )

# Step 3: Second aggregation (causes another Exchange)
final_df = agg_df.groupBy() \
                 .agg(_sum("total_value").alias("grand_total"))

# Action (triggers execution)
final_df.show()

+-----------+
|grand_total|
+-----------+
|        195|
+-----------+



In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as _sum, count
from pyspark.sql.types import IntegerType
import random

spark = SparkSession.builder \
    .appName("Strong_DAG_Example") \
    .getOrCreate()

spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.shuffle.partitions", "200")

# Increase shuffle partitions to show more tasks
spark.conf.set("spark.sql.shuffle.partitions", 200)

# Create large dataset (1 million rows)
data1 = [(random.randint(1, 1000), random.randint(1, 500)) for _ in range(1000000)]
data2 = [(i, f"Category_{i%50}") for i in range(1, 1001)]

df1 = spark.createDataFrame(data1, ["user_id", "amount"])
df2 = spark.createDataFrame(data2, ["user_id", "category"])

# Repartition to increase parallelism
df1 = df1.repartition(100)

# Join (Shuffle #1)
joined = df1.join(df2, "user_id")

# Aggregation (Shuffle #2)
agg = joined.groupBy("category") \
            .agg(
                count("*").alias("txn_count"),
                _sum("amount").alias("total_amount")
            )

# Global aggregation (Shuffle #3)
final = agg.groupBy() \
           .agg(_sum("total_amount").alias("grand_total"))

final.show()

+-----------+
|grand_total|
+-----------+
|  250288887|
+-----------+



----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 62730)
Traceback (most recent call last):
  File "C:\Users\onkar\AppData\Local\Programs\Python\Python39\lib\socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "C:\Users\onkar\AppData\Local\Programs\Python\Python39\lib\socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "C:\Users\onkar\AppData\Local\Programs\Python\Python39\lib\socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "C:\Users\onkar\AppData\Local\Programs\Python\Python39\lib\socketserver.py", line 720, in __init__
    self.handle()
  File "d:\data engineer\pyspark_practice\.venv\lib\site-packages\pyspark\accumulators.py", line 295, in handle
    poll(accum_updates)
  File "d:\data engineer\pyspark_practice\.venv\lib\site-packages\pyspark\accumulator

In [9]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, rand, concat, lit

spark = SparkSession.builder \
    .appName("OOM_Guaranteed") \
    .master("local[*]") \
    .config("spark.ui.enabled", "true") \
    .config("spark.ui.port", "4040") \
    .config("spark.driver.memory", "512m") \
    .config("spark.driver.maxResultSize", "256m") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("Spark UI --> http://localhost:4040")
print("Building large dataset ...")

df = spark.range(0, 50_000_000) \
    .withColumn("name",    concat(lit("customer_"), col("id").cast("string"))) \
    .withColumn("address", concat(lit("123 Street New York USA ZIP 10001 phone 555-010 email user"), col("id").cast("string"), lit("@gmail.com"))) \
    .withColumn("score",   rand() * 1000) \
    .withColumn("status",  concat(lit("status_active_premium_gold_tier_"), col("id").cast("string")))

print("Pulling 50 million rows to driver now ...")
print("Expected --> java.lang.OutOfMemoryError")
print()

# collect() pulls ALL 50M rows into driver JVM heap
# 50M rows x ~150 bytes = ~7.5GB into 512m driver heap

data = df.collect()

ConnectionRefusedError: [WinError 10061] No connection could be made because the target machine actively refused it

In [18]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ParquetTest") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

df = spark.read.option("mergeSchema", "true").parquet(
    r"D:\data engineer\pyspark_practice\pyspark_scenarios\dataset1_transactions\transactions"
)

df.show(5)


spark.stop()

Py4JJavaError: An error occurred while calling o173.parquet.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 6 in stage 4.0 failed 1 times, most recent failure: Lost task 6.0 in stage 4.0 (TID 19) (LAPTOP-CBUIVGN8 executor driver): org.apache.spark.sql.AnalysisException: Illegal Parquet type: INT64 (TIMESTAMP(NANOS,false)).
	at org.apache.spark.sql.errors.QueryCompilationErrors$.illegalParquetTypeError(QueryCompilationErrors.scala:1825)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkSchemaConverter.illegalType$1(ParquetSchemaConverter.scala:199)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkSchemaConverter.$anonfun$convertPrimitiveField$2(ParquetSchemaConverter.scala:276)
	at scala.Option.getOrElse(Option.scala:189)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkSchemaConverter.convertPrimitiveField(ParquetSchemaConverter.scala:217)
	at org.apache.spark.sql.execution.datasources.parquet.P